# 🦜🕸️ LangGraph Complete Masterclass

A comprehensive guide to building stateful, multi-actor applications with LangGraph - the graph-based framework for agents and workflows.

## Table of Contents
1. [Introduction & Core Concepts](#introduction)
2. [State & StateGraph](#state)
3. [Nodes & Edges](#nodes-edges)
4. [Conditional Logic](#conditional)
5. [Tool Calling & ReAct](#tool-calling)
6. [Multi-Agent Systems](#multi-agent)
7. [Memory & Persistence](#memory)
8. [Human-in-the-Loop](#human-loop)
9. [Streaming & Debugging](#streaming)
10. [Complete Example](#complete-example)

---
## 1. Introduction & Core Concepts <a id="introduction"></a>

### What is LangGraph?

LangGraph is a library for building **stateful, multi-actor applications** with LLMs. It extends LangChain's expression language with a graph-based approach.

```
┌─────────────────────────────────────────────────────────────┐
│              LangGraph Architecture                          │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│   ┌──────────┐                                              │
│   │  START   │ ──────────────────────────────────┐          │
│   └──────────┘                                   │          │
│        │                                         │          │
│        ▼                                         │          │
│   ┌──────────┐     ┌──────────┐     ┌──────────┐ │          │
│   │  Node A  │ ──▶ │  Node B  │ ──▶ │  Node C  │─┘          │
│   └──────────┘     └──────────┘     └──────────┘            │
│        │                                   │                 │
│        ▼                                   ▼                 │
│   ┌──────────┐     ┌──────────┐     ┌──────────┐            │
│   │  Node D  │ ──▶ │  Node E  │ ──▶ │   END    │            │
│   └──────────┘     └──────────┘     └──────────┘            │
│                                                              │
│   State flows through nodes, which transform it             │
└─────────────────────────────────────────────────────────────┘
```

### Key Concepts:

| Concept | Description | Example |
|---------|-------------|----------|
| **State** | Shared data structure passed between nodes | `dict`, `TypedDict` |
| **Node** | Function that transforms state | LLM call, tool execution |
| **Edge** | Connection between nodes | Always take next step |
| **Conditional Edge** | Dynamic routing based on state | Route to different nodes |
| **Graph** | Collection of nodes and edges | Complete workflow |

### Why LangGraph?

- **Cycles**: Unlike DAGs, LangGraph supports loops (essential for agents)
- **Statefulness**: Maintains context across multiple steps
- **Persistence**: Built-in checkpointing for memory
- **Human-in-the-Loop**: Pause and resume for human approval
- **Streaming**: Real-time output as the graph executes

In [ ]:
# Install LangGraph
%pip install langgraph langchain langchain-community langchain-ollama -q

In [ ]:
# Import core LangGraph modules
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, List
import operator

# Import LangChain components
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

# Initialize the model
llm = ChatOllama(
    model="qwen3.5:latest",
    temperature=0.7,
    base_url="http://localhost:11434"
)

print("✓ LangGraph initialized!")
print(f"  Model: qwen3.5:latest")
print(f"  Available: StateGraph, END, START, MemorySaver")

---
## 2. State & StateGraph <a id="state"></a>

### State Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    State Structure                           │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  State = TypedDict with annotated fields                    │
│                                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │ class State(TypedDict):                             │   │
│  │     messages: Annotated[List[Message], add_messages]│   │
│  │     count: int                                      │   │
│  │     data: dict                                      │   │
│  │     └── Each field has a "reducer" function        │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                              │
│  Reducers define how state updates when nodes return values │
│  • Default: Replace old value with new value                │
│  • operator.add: Append/concatenate                         │
│  • Custom function: Custom merge logic                      │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Defining State with TypedDict ===

from typing import TypedDict, Annotated, List
import operator

# Simple state - just a dictionary
class SimpleState(TypedDict):
    """Simple state with basic fields"""
    input_text: str
    output_text: str
    step_count: int

# State with reducer - accumulates messages
class MessageState(TypedDict):
    """State that accumulates messages"""
    messages: Annotated[List[str], operator.add]  # Appends new messages
    summary: str  # Replaced on each update
    metadata: dict  # Replaced on each update

print("=== State Definitions ===")
print(f"SimpleState fields: {list(SimpleState.__annotations__.keys())}")
print(f"MessageState fields: {list(MessageState.__annotations__.keys())}")
print(f"  - messages: Uses 'operator.add' reducer (accumulates)")
print(f"  - summary: Default reducer (replaces)")
print(f"  - metadata: Default reducer (replaces)")

In [ ]:
# === Building a Simple StateGraph ===

from langgraph.graph import StateGraph, END, START

# Define the state
class TextProcessingState(TypedDict):
    original: str
    uppercase: str
    reversed: str
    word_count: int

# Define nodes (functions that transform state)
def to_uppercase(state: TextProcessingState) -> dict:
    """Convert text to uppercase"""
    return {"uppercase": state["original"].upper()}

def reverse_text(state: TextProcessingState) -> dict:
    """Reverse the text"""
    return {"reversed": state["original"][::-1]}

def count_words(state: TextProcessingState) -> dict:
    """Count words in the original text"""
    return {"word_count": len(state["original"].split())}

# Build the graph
builder = StateGraph(TextProcessingState)

# Add nodes
builder.add_node("uppercase", to_uppercase)
builder.add_node("reverse", reverse_text)
builder.add_node("count", count_words)

# Add edges (sequential flow)
builder.add_edge(START, "uppercase")  # Start -> uppercase
builder.add_edge("uppercase", "reverse")  # uppercase -> reverse
builder.add_edge("reverse", "count")  # reverse -> count
builder.add_edge("count", END)  # count -> End

# Compile the graph
graph = builder.compile()

print("=== Simple StateGraph ===")
print("Flow: START → uppercase → reverse → count → END")

# Run the graph
result = graph.invoke({"original": "Hello LangGraph World"})
print(f"\nResult:")
print(f"  Original: {result['original']}")
print(f"  Uppercase: {result['uppercase']}")
print(f"  Reversed: {result['reversed']}")
print(f"  Word Count: {result['word_count']}")

---
## 3. Nodes & Edges <a id="nodes-edges"></a>

### Node & Edge Types

```
┌─────────────────────────────────────────────────────────────┐
│                    Node Types                                │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌─────────────────┐                                        │
│  │ Function Node   │  def my_node(state): return {...}      │
│  │                 │  Returns dict to update state          │
│  └─────────────────┘                                        │
│                                                              │
│  ┌─────────────────┐                                        │
│  │ Runnable Node   │  prompt | llm | parser                 │
│  │ (LangChain)     │  Any LangChain Runnable                │
│  └─────────────────┘                                        │
│                                                              │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    Edge Types                                │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌─────────────────┐                                        │
│  │ Normal Edge     │  Always goes from A to B              │
│  │ (add_edge)      │  graph.add_edge("A", "B")              │
│  └─────────────────┘                                        │
│                                                              │
│  ┌─────────────────┐                                        │
│  │ Conditional     │  Routes based on state                │
│  │ (add_conditional│  graph.add_conditional_edges(         │
│  │  _edges)        │      "A", route_fn, {"yes": "B", ...})│
│  └─────────────────┘                                        │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Different Node Types ===

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class AnalysisState(TypedDict):
    topic: str
    summary: str
    sentiment: str
    keywords: List[str]

# Node 1: Function-based node
def extract_topic(state: AnalysisState) -> dict:
    """Extract and normalize the topic"""
    return {"topic": state["topic"].strip().title()}

# Node 2: LLM-based node (Runnable)
summary_prompt = ChatPromptTemplate.from_template(
    "Write a one-sentence summary about: {topic}"
)
summary_chain = summary_prompt | llm | StrOutputParser()

def generate_summary(state: AnalysisState) -> dict:
    """Generate summary using LLM"""
    result = summary_chain.invoke({"topic": state["topic"]})
    return {"summary": result}

# Node 3: Another function node
def analyze_sentiment(state: AnalysisState) -> dict:
    """Simple keyword-based sentiment analysis"""
    positive_words = ["good", "great", "excellent", "amazing", "love"]
    negative_words = ["bad", "terrible", "awful", "hate", "worst"]
    
    topic_lower = state["topic"].lower()
    if any(word in topic_lower for word in positive_words):
        return {"sentiment": "positive"}
    elif any(word in topic_lower for word in negative_words):
        return {"sentiment": "negative"}
    return {"sentiment": "neutral"}

# Build graph
builder = StateGraph(AnalysisState)
builder.add_node("extract", extract_topic)
builder.add_node("summarize", generate_summary)
builder.add_node("sentiment", analyze_sentiment)

builder.add_edge(START, "extract")
builder.add_edge("extract", "summarize")
builder.add_edge("summarize", "sentiment")
builder.add_edge("sentiment", END)

analysis_graph = builder.compile()

print("=== Analysis Graph ===")
print("Flow: START → extract → summarize → sentiment → END")

# Run
result = analysis_graph.invoke({"topic": "the amazing benefits of exercise"})
print(f"\nTopic: {result['topic']}")
print(f"Summary: {result['summary']}")
print(f"Sentiment: {result['sentiment']}")

In [ ]:
# === Conditional Edges - Dynamic Routing ===

class RouterState(TypedDict):
    query: str
    response: str
    category: str

# Router function - decides which path to take
def route_query(state: RouterState) -> str:
    """Route based on query content"""
    query_lower = state["query"].lower()
    
    if any(word in query_lower for word in ["math", "calculate", "number", "plus", "minus"]):
        return "math_route"
    elif any(word in query_lower for word in ["weather", "temperature", "forecast"]):
        return "weather_route"
    elif any(word in query_lower for word in ["joke", "funny", "laugh"]):
        return "joke_route"
    else:
        return "general_route"

# Define handler nodes
def handle_math(state: RouterState) -> dict:
    return {"response": f"[Math Handler] Processing: {state['query']}"}

def handle_weather(state: RouterState) -> dict:
    return {"response": f"[Weather Handler] Looking up: {state['query']}"}

def handle_joke(state: RouterState) -> dict:
    return {"response": f"[Joke Handler] Here's a joke for: {state['query']}"}

def handle_general(state: RouterState) -> dict:
    result = llm.invoke(state["query"]).content
    return {"response": result, "category": "general"}

# Build graph with conditional edges
builder = StateGraph(RouterState)

# Add nodes
builder.add_node("math", handle_math)
builder.add_node("weather", handle_weather)
builder.add_node("joke", handle_joke)
builder.add_node("general", handle_general)

# Add START edge
builder.add_edge(START, "router")

# Add conditional edges
builder.add_conditional_edges(
    "router",           # Source node
    route_query,        # Routing function
    {
        "math_route": "math",
        "weather_route": "weather",
        "joke_route": "joke",
        "general_route": "general"
    }
)

# All handlers go to END
builder.add_edge("math", END)
builder.add_edge("weather", END)
builder.add_edge("joke", END)
builder.add_edge("general", END)

router_graph = builder.compile()

print("=== Router Graph ===")
print("Routes: math, weather, joke, general")

# Test different queries
test_queries = [
    "Calculate 25 + 17",
    "What's the weather like?",
    "Tell me something funny",
    "Explain quantum physics"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    result = router_graph.invoke({"query": query})
    print(f"Response: {result['response'][:80]}...")

---
## 4. Conditional Logic <a id="conditional"></a>

### Conditional Edge Patterns

```
┌─────────────────────────────────────────────────────────────┐
│              Conditional Edge Patterns                       │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Pattern 1: Router                                          │
│  ┌──────────┐                                               │
│  │  Router  │──▶ math ──▶ END                              │
│  │  Node    │──▶ weather ──▶ END                           │
│  └──────────┘──▶ general ──▶ END                           │
│                                                              │
│  Pattern 2: Loop with Condition                             │
│  ┌──────────┐     ┌──────────┐                              │
│  │  Process │ ──▶ │  Check   │ ──▶ END (if done)          │
│  └──────────┘     └──────────┘                              │
│       ▲                  │                                  │
│       │                  │ (if not done)                    │
│       └──────────────────┘                                  │
│                                                              │
│  Pattern 3: Branch and Merge                                │
│         ┌──▶ Node A ──┐                                     │
│  Start ─┼──▶ Node B ──┼──▶ Merge ──▶ END                  │
│         └──▶ Node C ──┘                                     │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Loop Pattern - Agent-style Iteration ===

class LoopState(TypedDict):
    question: str
    answer: str
    attempts: int
    is_correct: bool

# LLM-based answer generator
answer_prompt = ChatPromptTemplate.from_template(
    """You are solving a math problem. Think step by step.

Question: {question}
Previous attempts: {attempts}

Your answer:"""
)
answer_chain = answer_prompt | llm | StrOutputParser()

def generate_answer(state: LoopState) -> dict:
    """Generate an answer attempt"""
    answer = answer_chain.invoke({
        "question": state["question"],
        "attempts": state["attempts"]
    })
    return {
        "answer": answer,
        "attempts": state["attempts"] + 1
    }

def check_answer(state: LoopState) -> str:
    """Check if we should continue or stop"""
    # Simple heuristic: stop after 3 attempts
    if state["attempts"] >= 3:
        return "done"
    return "retry"

# Build the loop graph
builder = StateGraph(LoopState)

builder.add_node("generate", generate_answer)
builder.add_node("check", lambda s: {})

builder.add_edge(START, "generate")
builder.add_conditional_edges(
    "generate",
    check_answer,
    {
        "retry": "check",  # Loop back
        "done": END
    }
)
builder.add_edge("check", "generate")

loop_graph = builder.compile()

print("=== Loop Graph ===")
print("Flow: START → generate → check → (retry|done)")

# Run
result = loop_graph.invoke({
    "question": "What is 15% of 80?",
    "answer": "",
    "attempts": 0,
    "is_correct": False
})

print(f"\nQuestion: {result['question']}")
print(f"Final Answer: {result['answer'][:100]}...")
print(f"Attempts: {result['attempts']}")

In [ ]:
# === Branch and Merge Pattern ===

class BranchState(TypedDict):
    topic: str
    outline: str
    introduction: str
    conclusion: str
    full_article: str

# Generate outline
def create_outline(state: BranchState) -> dict:
    prompt = f"Create a 3-point outline for an article about: {state['topic']}"
    outline = llm.invoke(prompt).content
    return {"outline": outline}

# Write introduction
def write_intro(state: BranchState) -> dict:
    prompt = f"Write an engaging introduction for an article about: {state['topic']}"
    intro = llm.invoke(prompt).content
    return {"introduction": intro}

# Write conclusion
def write_conclusion(state: BranchState) -> dict:
    prompt = f"Write a compelling conclusion for an article about: {state['topic']}"
    conclusion = llm.invoke(prompt).content
    return {"conclusion": conclusion}

# Merge all parts
def merge_article(state: BranchState) -> dict:
    full = f"""
{state['introduction']}

**Outline:**
{state['outline']}

{state['conclusion']}
"""
    return {"full_article": full}

# Build graph
builder = StateGraph(BranchState)

builder.add_node("outline", create_outline)
builder.add_node("intro", write_intro)
builder.add_node("conclusion", write_conclusion)
builder.add_node("merge", merge_article)

# Branch out from START
builder.add_edge(START, "outline")
builder.add_edge("outline", "intro")
builder.add_edge("intro", "conclusion")
builder.add_edge("conclusion", "merge")
builder.add_edge("merge", END)

article_graph = builder.compile()

print("=== Article Generator Graph ===")
print("Flow: START → outline → intro → conclusion → merge → END")

# Run
result = article_graph.invoke({"topic": "The Future of AI in Healthcare"})

print(f"\n=== Generated Article ===")
print(result['full_article'][:500]}...)

---
## 5. Tool Calling & ReAct Agent <a id="tool-calling"></a>

### ReAct Agent Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                 ReAct Agent Loop                             │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌──────────┐                                               │
│  │   START  │                                               │
│  └────┬─────┘                                               │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────┐     Yes     ┌─────────────┐                │
│  │  LLM Call   │ ──────────▶ │ Call Tool   │                │
│  │ (with tools)│             │ & Store     │                │
│  └──────┬──────┘             └──────┬──────┘                │
│         │ No                        │                        │
│         │  Has Tool Call            │                        │
│         │         ┌─────────────────┘                        │
│         ▼         ▼                                          │
│  ┌─────────────────────────┐                                 │
│  │   Generate Final        │                                 │
│  │   Response              │                                 │
│  └─────────────────────────┘                                 │
│                                                              │
│  State accumulates: messages + tool calls + observations    │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Building a Tool-Calling Agent ===

from typing import Annotated
from langchain_core.messages import BaseMessage, add_messages
import json

# Define tools
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    # Simulated search results
    results = {
        "python": "Python is a high-level programming language created by Guido van Rossum.",
        "langchain": "LangChain is a framework for developing applications powered by language models.",
        "weather": "Current weather: 22°C, Partly Cloudy in most regions.",
    }
    query_lower = query.lower()
    for key, value in results.items():
        if key in query_lower:
            return value
    return f"No specific results found for '{query}'. Try: python, langchain, weather"

@tool
def calculate(expression: str) -> str:
    """Evaluate mathematical expressions."""
    try:
        # Safe eval for basic math
        allowed = set("0123456789+-*/.() ")
        if all(c in allowed for c in expression):
            result = eval(expression)
            return f"{expression} = {result}"
        return "Invalid characters"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_time(location: str) -> str:
    """Get the current time for a location."""
    times = {
        "london": "14:30 GMT",
        "tokyo": "23:30 JST",
        "new york": "09:30 EST",
    }
    return times.get(location.lower(), f"Time data not available for {location}")

tools = [search_web, calculate, get_time]
tool_map = {tool.name: tool for tool in tools}

# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

print("=== Tools Available ===")
for name, tool_obj in tool_map.items():
    print(f"  - {name}: {tool_obj.description}")

In [ ]:
# === Define Agent State ===

from typing import Annotated
from langchain_core.messages import BaseMessage, add_messages

class AgentState(TypedDict):
    """State for the tool-calling agent"""
    messages: Annotated[List[BaseMessage], add_messages]

# Define nodes
def agent_node(state: AgentState) -> dict:
    """LLM decides what to do"""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def tool_node(state: AgentState) -> dict:
    """Execute tools based on LLM's decision"""
    messages = state["messages"]
    last_message = messages[-1]
    
    # Check for tool calls
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        results = []
        for tool_call in last_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            if tool_name in tool_map:
                result = tool_map[tool_name].invoke(tool_args)
                results.append({
                    "role": "tool",
                    "content": str(result),
                    "tool_call_id": tool_call.get("id", "")
                })
        return {"messages": results}
    
    return {"messages": []}

# Router function
def should_continue(state: AgentState) -> str:
    """Decide whether to call tools or end"""
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    return "end"

# Build the agent graph
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)
builder.add_edge("tools", "agent")  # Loop back after tool execution

agent_graph = builder.compile()

print("=== Tool-Calling Agent ===")
print("Flow: START → agent → (tools → agent)* → END")

In [ ]:
# === Test the Agent ===

from langchain_core.messages import HumanMessage

def run_agent(query: str):
    """Run the agent with a query"""
    initial_state = {
        "messages": [HumanMessage(content=query)]
    }
    result = agent_graph.invoke(initial_state)
    return result["messages"][-1].content

print("=== Agent Test Cases ===")

# Test 1: Simple query (no tools)
print("\n1. Simple greeting:")
response = run_agent("Hello! How are you?")
print(f"   Response: {response[:100]}...")

# Test 2: Web search
print("\n2. Web search:")
response = run_agent("What is Python?")
print(f"   Response: {response[:150]}...")

# Test 3: Calculation
print("\n3. Math calculation:")
response = run_agent("Calculate 125 * 8 + 50")
print(f"   Response: {response[:100]}...")

# Test 4: Multi-step (requires multiple tool calls)
print("\n4. Multi-step query:")
response = run_agent("What is Python and what time is it in Tokyo?")
print(f"   Response: {response[:200]}...")

---
## 6. Multi-Agent Systems <a id="multi-agent"></a>

### Multi-Agent Architecture

```
┌─────────────────────────────────────────────────────────────┐
│              Multi-Agent Supervisor Pattern                  │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│                      ┌─────────────┐                        │
│                      │ Supervisor  │                        │
│                      │   (LLM)     │                        │
│                      └──────┬──────┘                        │
│                             │                                │
│           ┌─────────────────┼─────────────────┐             │
│           │                 │                 │              │
│           ▼                 ▼                 ▼              │
│    ┌────────────┐   ┌────────────┐   ┌────────────┐         │
│    │ Researcher │   │  Coder     │   │  Writer    │         │
│    │   Agent    │   │   Agent    │   │   Agent    │         │
│    └─────┬──────┘   └─────┬──────┘   └─────┬──────┘         │
│          │                │                │                 │
│          └────────────────┼────────────────┘                 │
│                           │                                  │
│                           ▼                                  │
│                    ┌─────────────┐                           │
│                    │   Merge     │                           │
│                    │  Results    │                           │
│                    └─────────────┘                           │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Multi-Agent System with Supervisor ===

from typing import Literal

class MultiAgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    task: str
    current_agent: str
    final_response: str

# Define specialized agents
researcher_prompt = ChatPromptTemplate.from_template(
    """You are a Researcher. Find and summarize information about the topic.

Topic: {task}

Provide key facts and findings:"""
)

coder_prompt = ChatPromptTemplate.from_template(
    """You are a Coder. Write code examples or solve programming tasks.

Task: {task}

Provide working code with explanations:"""
)

writer_prompt = ChatPromptTemplate.from_template(
    """You are a Writer. Create well-written content.

Topic: {task}

Write clear, engaging content:"""
)

def researcher_node(state: MultiAgentState) -> dict:
    response = researcher_prompt.invoke({"task": state["task"]})
    response = llm.invoke(response)
    return {
        "messages": [AIMessage(content=f"[Researcher]: {response.content}")],
        "final_response": response.content
    }

def coder_node(state: MultiAgentState) -> dict:
    response = coder_prompt.invoke({"task": state["task"]})
    response = llm.invoke(response)
    return {
        "messages": [AIMessage(content=f"[Coder]: {response.content}")],
        "final_response": response.content
    }

def writer_node(state: MultiAgentState) -> dict:
    response = writer_prompt.invoke({"task": state["task"]})
    response = llm.invoke(response)
    return {
        "messages": [AIMessage(content=f"[Writer]: {response.content}")],
        "final_response": response.content
    }

def supervisor_node(state: MultiAgentState) -> dict:
    """Supervisor decides which agent should handle the task"""
    
    supervisor_prompt = f"""You are a supervisor. Route the task to the best agent.

Task: {state['task']}

Choose: researcher, coder, or writer

Respond with ONLY the agent name:"""
    
    response = llm.invoke(supervisor_prompt)
    agent = response.content.strip().lower()
    
    # Map to valid node names
    if "research" in agent:
        agent = "researcher"
    elif "code" in agent or "program" in agent:
        agent = "coder"
    elif "write" in agent or "content" in agent:
        agent = "writer"
    else:
        agent = "writer"  # Default
    
    return {"current_agent": agent}

def route_from_supervisor(state: MultiAgentState) -> str:
    return state.get("current_agent", "writer")

# Build multi-agent graph
builder = StateGraph(MultiAgentState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("coder", coder_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    ["researcher", "coder", "writer"]
)
builder.add_edge("researcher", END)
builder.add_edge("coder", END)
builder.add_edge("writer", END)

multi_agent_graph = builder.compile()

print("=== Multi-Agent System ===")
print("Agents: Researcher, Coder, Writer")
print("Flow: START → supervisor → [agent] → END")

In [ ]:
# === Test Multi-Agent System ===

def run_multi_agent(task: str):
    initial_state = {
        "messages": [],
        "task": task,
        "current_agent": "",
        "final_response": ""
    }
    result = multi_agent_graph.invoke(initial_state)
    return result

test_tasks = [
    "Explain how photosynthesis works",
    "Write a Python function to sort a list",
    "Create a product description for a smartwatch"
]

print("=== Multi-Agent Test ===")
for task in test_tasks:
    print(f"\nTask: {task}")
    result = run_multi_agent(task)
    print(f"Assigned to: {result['current_agent']}")
    print(f"Response: {result['final_response'][:150]}...")

---
## 7. Memory & Persistence <a id="memory"></a>

### Checkpointing Architecture

```
┌─────────────────────────────────────────────────────────────┐
│              Persistence with Checkpoints                    │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Thread 1:                        Thread 2:                 │
│  ┌─────────────┐                  ┌─────────────┐           │
│  │ Checkpoint  │                  │ Checkpoint  │           │
│  │     A       │                  │     C       │           │
│  └──────┬──────┘                  └──────┬──────┘           │
│         │                                │                   │
│         ▼                                ▼                   │
│  ┌─────────────┐                  ┌─────────────┐           │
│  │ Checkpoint  │                  │ Checkpoint  │           │
│  │     B       │                  │     D       │           │
│  └─────────────┘                  └─────────────┘           │
│                                                              │
│  Checkpointers:                                             │
│  • MemorySaver - In-memory (development)                    │
│  • SqliteSaver - SQLite database (production)               │
│  • RedisSaver - Redis (distributed)                         │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Memory with Checkpointer ===

from langgraph.checkpoint.memory import MemorySaver
from uuid import uuid4

# Simple conversational state
class ConversationState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    user_name: str
    user_preferences: dict

# Conversation node
def conversation_node(state: ConversationState) -> dict:
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

# Build graph with memory
builder = StateGraph(ConversationState)
builder.add_node("conversation", conversation_node)
builder.add_edge(START, "conversation")
builder.add_edge("conversation", END)

# Add memory saver
memory = MemorySaver()
conversation_graph = builder.compile(checkpointer=memory)

print("=== Conversation with Memory ===")
print("Using MemorySaver for persistence")

# Generate a thread ID (like a session)
thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

print(f"\nThread ID: {thread_id[:8]}...")

In [ ]:
# === Multi-turn Conversation ===

def chat(message: str, config: dict):
    """Send a message and get response"""
    state = {"messages": [HumanMessage(content=message)]}
    result = conversation_graph.invoke(state, config)
    return result["messages"][-1].content

print("=== Multi-turn Conversation ===")

# First exchange
print("\nUser: Hi, my name is Alice")
response = chat("Hi, my name is Alice", config)
print(f"AI: {response[:100]}...")

# Second exchange (should remember name)
print("\nUser: What's my name?")
response = chat("What's my name?", config)
print(f"AI: {response[:100]}...")

# Third exchange
print("\nUser: I like pizza and pasta")
response = chat("I like pizza and pasta", config)
print(f"AI: {response[:100]}...")

# Fourth exchange (should remember food preferences)
print("\nUser: What food do I like?")
response = chat("What food do I like?", config)
print(f"AI: {response[:100]}...")

# New thread (no memory of previous conversation)
print("\n--- New Thread (Fresh Memory) ---")
new_config = {"configurable": {"thread_id": str(uuid4())}}
print("\nUser: What's my name?")
response = chat("What's my name?", new_config)
print(f"AI: {response[:100]}...")

---
## 8. Human-in-the-Loop <a id="human-loop"></a>

### Human Approval Pattern

```
┌─────────────────────────────────────────────────────────────┐
│              Human-in-the-Loop Flow                          │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌──────────┐                                               │
│  │   LLM    │                                               │
│  │ Suggests │                                               │
│  └────┬─────┘                                               │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────────┐     Approve    ┌─────────────┐         │
│  │  Interrupt /   │ ──────────────▶ │  Continue   │         │
│  │  Pause Graph   │                 │  Execution  │         │
│  └─────────────────┘                 └─────────────┘         │
│         │                                                    │
│         │ Reject/Modify                                    │
│         ▼                                                    │
│  ┌─────────────────┐                                        │
│  │  Human Edits    │                                        │
│  │  State/Action   │                                        │
│  └─────────────────┘                                        │
│                                                              │
│  Use Cases:                                                 │
│  • Code review before deployment                            │
│  • Content approval before publishing                       │
│  • Sensitive action confirmation                            │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Human-in-the-Loop with Interruption ===

class ApprovalState(TypedDict):
    task: str
    draft: str
    approved: bool
    feedback: str
    final: str

# Draft generation
def generate_draft(state: ApprovalState) -> dict:
    prompt = f"""Write a professional email about: {state['task']}
    
    Keep it concise and clear:"""
    draft = llm.invoke(prompt).content
    return {"draft": draft}

# Human approval node (simulated)
def human_approval(state: ApprovalState) -> dict:
    print("\n" + "="*50)
    print("📋 HUMAN REVIEW REQUIRED")
    print("="*50)
    print(f"\nDraft:\n{state['draft']}")
    print("\n" + "="*50)
    
    # Simulated approval (in real app, wait for user input)
    # For demo, auto-approve
    approved = True
    feedback = "Looks good!"
    
    print(f"Decision: {'APPROVED' if approved else 'REJECTED'}")
    print(f"Feedback: {feedback}")
    print("="*50 + "\n")
    
    return {
        "approved": approved,
        "feedback": feedback,
        "final": state["draft"] if approved else None
    }

# Check approval status
def check_approval(state: ApprovalState) -> str:
    if state.get("approved", False):
        return "approved"
    return "needs_revision"

# Revision node
def revise_draft(state: ApprovalState) -> dict:
    prompt = f"""Revise this draft based on feedback:
    
    Original: {state['draft']}
    Feedback: {state['feedback']}
    
    Revised version:"""
    revised = llm.invoke(prompt).content
    return {"draft": revised, "final": revised}

# Build graph
builder = StateGraph(ApprovalState)

builder.add_node("draft", generate_draft)
builder.add_node("review", human_approval)
builder.add_node("revise", revise_draft)

builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_conditional_edges(
    "review",
    check_approval,
    {
        "approved": END,
        "needs_revision": "revise"
    }
)
builder.add_edge("revise", END)

# Compile with interrupt before review
memory = MemorySaver()
approval_graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["review"]
)

print("=== Human-in-the-Loop Graph ===")
print("Flow: START → draft → [INTERRUPT] → review → (approved|revise) → END")
print("\nGraph will pause before 'review' node for human approval")

In [ ]:
# === Run with Interruption ===

thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

# Start the graph
print("=== Starting Approval Workflow ===")
initial_state = {"task": "Request a meeting to discuss Q2 goals"}

# Run until interruption
result = approval_graph.invoke(initial_state, config)
print(f"\n⏸️  Graph paused at: review node")
print(f"Draft generated: {result.get('draft', 'N/A')[:100]}...")

# In real app, user would review and approve here
# For demo, we continue
print("\n✅ Simulating human approval...")

# Continue after approval
result = approval_graph.invoke(None, config)
print(f"\n✅ Final approved draft:")
print(f"{result.get('final', 'N/A')[:200]}...")

---
## 9. Streaming & Debugging <a id="streaming"></a>

### Streaming Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Streaming Events                          │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Event Types:                                               │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐      │
│  │ on_chain_start│  │ on_chain_end│  │ on_token    │      │
│  │              │  │              │  │ (streaming)  │      │
│  └──────────────┘  └──────────────┘  └──────────────┘      │
│                                                              │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐      │
│  │ on_node_start│  │ on_node_end │  │ on_custom   │      │
│  │              │  │              │  │ event       │      │
│  └──────────────┘  └──────────────┘  └──────────────┘      │
│                                                              │
│  Streaming Modes:                                           │
│  • stream() - Iterate over events as they happen           │
│  • astream() - Async streaming                              │
│  • stream_events() - Fine-grained event control            │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Streaming Output ===

class StreamState(TypedDict):
    topic: str
    content: str

def generate_content(state: StreamState) -> dict:
    prompt = f"Write a detailed paragraph about: {state['topic']}"
    content = llm.invoke(prompt).content
    return {"content": content}

builder = StateGraph(StreamState)
builder.add_node("generate", generate_content)
builder.add_edge(START, "generate")
builder.add_edge("generate", END)

stream_graph = builder.compile()

print("=== Streaming Demo ===")
print("\nGenerating content with streaming...\n")

# Stream the output
for event in stream_graph.stream({"topic": "artificial intelligence"}):
    for node_name, output in event.items():
        print(f"\n[{node_name}] completed:")
        print(f"  Content: {output.get('content', 'N/A')[:100]}...")

In [ ]:
# === Streaming with Events ===

print("=== Event-Based Streaming ===")

# Create a more complex graph for demonstration
class DebugState(TypedDict):
    input: str
    step1: str
    step2: str
    final: str

def step1_node(state: DebugState) -> dict:
    result = llm.invoke(f"Analyze: {state['input']}").content
    return {"step1": result}

def step2_node(state: DebugState) -> dict:
    result = llm.invoke(f"Expand on: {state['step1']}").content
    return {"step2": result}

def final_node(state: DebugState) -> dict:
    result = llm.invoke(f"Summarize: {state['step2']}").content
    return {"final": result}

builder = StateGraph(DebugState)
builder.add_node("analyze", step1_node)
builder.add_node("expand", step2_node)
builder.add_node("summarize", final_node)

builder.add_edge(START, "analyze")
builder.add_edge("analyze", "expand")
builder.add_edge("expand", "summarize")
builder.add_edge("summarize", END)

debug_graph = builder.compile()

print("\nStreaming each node execution:\n")

for event in debug_graph.stream({"input": "machine learning"}, stream_mode="values"):
    print(f"\n--- State Update ---")
    for key, value in event.items():
        if value:
            print(f"  {key}: {str(value)[:80]}...")

---
## 10. Complete Example <a id="complete-example"></a>

### Research Assistant with All Features

```
┌─────────────────────────────────────────────────────────────┐
│           Complete Research Assistant                        │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌──────────┐                                               │
│  │  START   │                                               │
│  └────┬─────┘                                               │
│       │                                                     │
│       ▼                                                     │
│  ┌─────────────┐                                           │
│  │  Planner    │ ── Creates research plan                  │
│  └──────┬──────┘                                           │
│         │                                                   │
│         ▼                                                   │
│  ┌─────────────┐     ┌─────────────┐                       │
│  │  Research   │ ──▶ │   Review    │ ──▶ [Human Approval] │
│  │   Agent     │     │   (Check)   │                       │
│  └─────────────┘     └─────────────┘                       │
│         │                                                   │
│         ▼                                                   │
│  ┌─────────────┐                                           │
│  │   Writer    │ ── Compiles final report                 │
│  └──────┬──────┘                                           │
│         │                                                   │
│         ▼                                                   │
│  ┌─────────────┐                                           │
│  │    END      │                                           │
│  └─────────────┘                                           │
│                                                              │
│  Features: Tools + Multi-Agent + Memory + Human-in-Loop    │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Complete Research Assistant ===

from typing import Annotated
from langchain_core.messages import HumanMessage, AIMessage, add_messages

# Research tools
@tool
def search_academic(query: str) -> str:
    """Search academic papers and research."""
    papers = {
        "ai": "Key AI papers: Attention Is All You Need (2017), GPT-4 Technical Report (2023)",
        "ml": "ML fundamentals: Deep Learning by Goodfellow (2016), Pattern Recognition by Bishop",
        "nlp": "NLP advances: BERT (2018), T5 (2019), LLaMA (2023)",
    }
    for key, value in papers.items():
        if key in query.lower():
            return value
    return "General research available. Try: ai, ml, nlp"

@tool
def get_statistics(topic: str) -> str:
    """Get statistics about a topic."""
    stats = {
        "ai": "AI market: $200B (2023), projected $1.8T by 2030",
        "ml": "ML adoption: 35% of companies using ML (2023)",
    }
    for key, value in stats.items():
        if key in topic.lower():
            return value
    return "Statistics available for: ai, ml"

research_tools = [search_academic, get_statistics]
tool_map = {t.name: t for t in research_tools}
llm_with_tools = llm.bind_tools(research_tools)

# State definition
class ResearchState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    topic: str
    plan: str
    research_findings: str
    draft: str
    final_report: str
    approved: bool

print("=== Research Assistant ===")
print(f"Tools: {[t.name for t in research_tools]}")

In [ ]:
# === Define Nodes ===

def planner_node(state: ResearchState) -> dict:
    """Create a research plan"""
    prompt = f"""Create a research plan for: {state['topic']}
    
    Include:
    1. Key questions to answer
    2. Areas to investigate
    3. Expected output format
    
    Plan:"""
    plan = llm.invoke(prompt).content
    return {"plan": plan}

def researcher_node(state: ResearchState) -> dict:
    """Conduct research using tools"""
    messages = state["messages"] + [HumanMessage(content=f"Research: {state['topic']}")]
    
    # Run agent loop (simplified)
    response = llm_with_tools.invoke(messages)
    findings = response.content
    
    # Execute tool calls if any
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            if tool_name in tool_map:
                result = tool_map[tool_name].invoke(tool_call["args"])
                findings += f"\n\n[Tool: {tool_name}]\n{result}"
    
    return {"research_findings": findings}

def writer_node(state: ResearchState) -> dict:
    """Write the final report"""
    prompt = f"""Write a comprehensive research report.
    
    Topic: {state['topic']}
    Plan: {state['plan']}
    Findings: {state['research_findings']}
    
    Format:
    - Executive Summary
    - Key Findings
    - Conclusions
    
    Report:"""
    report = llm.invoke(prompt).content
    return {"final_report": report}

def review_node(state: ResearchState) -> dict:
    """Human review point"""
    print("\n" + "="*60)
    print("📋 RESEARCH REPORT READY FOR REVIEW")
    print("="*60)
    print(f"\n{state['final_report'][:500]}...")
    print("\n" + "="*60)
    
    # Simulated approval
    return {"approved": True}

def check_review(state: ResearchState) -> str:
    return "approved" if state.get("approved") else "rejected"

# Build the complete graph
builder = StateGraph(ResearchState)

builder.add_node("planner", planner_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)
builder.add_node("review", review_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "review")
builder.add_conditional_edges(
    "review",
    check_review,
    {
        "approved": END,
        "rejected": "researcher"  # Loop back if rejected
    }
)

memory = MemorySaver()
research_graph = builder.compile(checkpointer=memory)

print("\n=== Research Assistant Graph ===")
print("Flow: START → planner → researcher → writer → review → END")

In [ ]:
# === Run Complete Research Assistant ===

def run_research(topic: str):
    thread_id = str(uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    
    initial_state = {
        "messages": [HumanMessage(content=f"Research: {topic}")],
        "topic": topic,
        "plan": "",
        "research_findings": "",
        "draft": "",
        "final_report": "",
        "approved": False
    }
    
    print(f"\n🔬 Starting research on: {topic}\n")
    
    # Stream execution
    for event in research_graph.stream(initial_state, config, stream_mode="values"):
        if "final_report" in event and event["final_report"]:
            print("\n📝 Report generated...")
    
    # Get final result
    result = research_graph.invoke(None, config)
    return result

# Run the research assistant
result = run_research("artificial intelligence in healthcare")

print("\n" + "="*60)
print("✅ FINAL RESEARCH REPORT")
print("="*60)
print(result["final_report"])

---

## Summary & Next Steps

### You've Learned:

| Component | Key Classes | Use Case |
|-----------|-------------|----------|
| **State** | `TypedDict`, `Annotated` | Shared data structure |
| **Graph** | `StateGraph`, `START`, `END` | Define workflow |
| **Nodes** | Functions, Runnables | Transform state |
| **Edges** | `add_edge`, `add_conditional_edges` | Control flow |
| **Tools** | `@tool`, `bind_tools` | External functions |
| **Memory** | `MemorySaver`, `checkpointer` | Persistence |
| **HITL** | `interrupt_before` | Human approval |
| **Streaming** | `stream()`, `stream_mode` | Real-time output |

### Graph Patterns:

```
1. Sequential:     A → B → C
2. Conditional:    A → (B|C|D) based on state
3. Loop:           A → B → (A|END)
4. Multi-Agent:    Supervisor → [Agent1, Agent2, Agent3]
5. HITL:           A → [INTERRUPT] → Human → B
```

### Next Steps:
1. Explore `src/agents/` for production-ready agent implementations
2. Check `langgraph_advanced.ipynb` for subgraphs and cross-graph communication
3. Build your own agent in the `projects/` folder

### Resources:
- LangGraph Docs: https://langchain-ai.github.io/langgraph/
- LangChain Docs: https://python.langchain.com/
- GitHub: https://github.com/langchain-ai/langgraph